# R1-05 · SQL para análisis — Profundización (opcional) 🔬

**Formación Pública — Rama R1 · Analista de Datos · Notebook de profundización**

Opcional y de **transferencia**: el mismo SQL sobre otro dominio — los **24 parques nacionales**
de Chile (CONAF / SNASPE: nombre, región, año de declaración y superficie). Herramientas nuevas:
`LIKE`, **subconsultas**, `CASE WHEN` y `WHERE` + `HAVING` juntos (el orden lógico de SQL).

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, os, urllib.request, sqlite3
CSV = "parques.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/parques.csv", CSV)
df = pd.read_csv(CSV)            # 24 parques nacionales (CONAF / SNASPE)
conn = sqlite3.connect(":memory:")
df.to_sql("parques", conn, index=False, if_exists="replace")

def consultar(sql):
    return pd.read_sql_query(sql, conn)

consultar("SELECT * FROM parques LIMIT 5")

## 1. LIKE: buscar texto parcial

`LIKE` busca patrones: `%` significa «cualquier cosa». `nombre LIKE 'Laguna%'` = los que empiezan con «Laguna». Ordénalos por año de declaración.

### ✍️ Ejercicio 1 — Parques cuyo nombre empieza con «Laguna»

In [ ]:
sql_01 = "SELECT nombre, anio FROM parques WHERE nombre LIKE 'Laguna%' ORDER BY anio"
consultar(sql_01)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    r = consultar(sql_01)
    _esp = df[df["nombre"].str.startswith("Laguna")].sort_values("anio")
    assert list(r["nombre"]) == list(_esp["nombre"])
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "WHERE nombre LIKE 'Laguna%' ORDER BY anio")
    print("   Detalle:", e)

## 2. GROUP BY + HAVING

¿Qué regiones tienen **3 o más** parques nacionales? Muestra cuántos y cuántas hectáreas suman.

### ✍️ Ejercicio 2 — Regiones con 3 o más parques

In [ ]:
sql_02 = """
SELECT region, COUNT(*) AS n_parques, SUM(superficie_ha) AS hectareas
FROM parques
GROUP BY region
HAVING COUNT(*) >= 3
ORDER BY n_parques DESC, hectareas DESC
"""
consultar(sql_02)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    r = consultar(sql_02)
    _c = df["region"].value_counts()
    assert set(r["region"]) == set(_c[_c >= 3].index)
    assert int(r["n_parques"].min()) >= 3
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'HAVING COUNT(*) >= 3')
    print("   Detalle:", e)

## 3. Subconsulta: comparar contra el promedio

Una consulta puede ir **dentro** de otra. `(SELECT AVG(superficie_ha) FROM parques)` devuelve un número que puedes usar en el `WHERE`. ¿Cuántos parques son más grandes que el parque «promedio»?

### ✍️ Ejercicio 3 — Parques sobre el promedio

In [ ]:
sql_03 = """
SELECT nombre, superficie_ha
FROM parques
WHERE superficie_ha > (SELECT AVG(superficie_ha) FROM parques)
ORDER BY superficie_ha DESC
"""
r = consultar(sql_03)
print(len(r), "de", len(df), "parques superan el promedio")
r

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    r = consultar(sql_03)
    assert len(r) == int((df["superficie_ha"] > df["superficie_ha"].mean()).sum())
    assert len(r) < len(df) / 2, "Pocos parques gigantes inflan el promedio: menos de la mitad lo supera"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'WHERE superficie_ha > (SELECT AVG(superficie_ha) FROM parques)')
    print("   Detalle:", e)

## 4. CASE WHEN: crear categorías dentro de la consulta

`CASE WHEN condición THEN valor ... ELSE valor END` crea una columna nueva, como un `SI()` de Excel. Clasifica cada parque por **época de declaración** y cuenta cuántos hay en cada una.

### ✍️ Ejercicio 4 — Parques por época

In [ ]:
sql_04 = """
SELECT CASE
         WHEN anio < 1960 THEN '1. antes de 1960'
         WHEN anio < 1990 THEN '2. 1960-1989'
         ELSE '3. 1990 en adelante'
       END AS epoca,
       COUNT(*) AS n
FROM parques
GROUP BY epoca
ORDER BY epoca
"""
consultar(sql_04)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    r = consultar(sql_04)
    _esp = [int((df["anio"] < 1960).sum()), int(((df["anio"] >= 1960) & (df["anio"] < 1990)).sum()), int((df["anio"] >= 1990).sum())]
    assert list(r["n"]) == _esp, f"Esperaba {_esp} parques por época"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "WHEN anio < 1990 THEN '2. 1960-1989' ELSE '3. 1990 en adelante' END")
    print("   Detalle:", e)

## 5. WHERE y HAVING en la misma consulta

El orden lógico de SQL es `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY`. Primero se filtran filas, después se agrupa, después se filtran grupos. Pregunta: considerando **solo los parques declarados antes de 1970**, ¿qué regiones ya protegían **más de 1 millón de hectáreas**?

### ✍️ Ejercicio 5 — Regiones con más de 1 millón de ha protegidas antes de 1970

In [ ]:
sql_05 = """
SELECT region, SUM(superficie_ha) AS hectareas
FROM parques
WHERE anio < 1970
GROUP BY region
HAVING SUM(superficie_ha) > 1000000
ORDER BY hectareas DESC
"""
consultar(sql_05)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    r = consultar(sql_05)
    _s = df[df["anio"] < 1970].groupby("region")["superficie_ha"].sum()
    assert set(r["region"]) == set(_s[_s > 1_000_000].index)
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'WHERE anio < 1970 ... HAVING SUM(superficie_ha) > 1000000')
    print("   Detalle:", e)

---

*Fin de la profundización de R1-05.*